In [1]:
import io
import pandas as pd

In [ ]:


def tootle_eesti_panga_intressid(failitee):
    # 1. Read the file, skipping the first 3 metadata rows.
    # FIXED: Removed the invalid 'index_code' parameter.
    df = pd.read_csv(failitee, sep=";", skiprows=3, encoding="ISO-8859-1")

    # Clean column names from unexpected trailing whitespaces
    df.columns = df.columns.str.strip()

    # Rename the very first blank column to "Laenuliik" so we can filter rows safely
    df.rename(columns={df.columns[0]: "Laenuliik"}, inplace=True)
    df["Laenuliik_Puhas"] = df["Laenuliik"].astype(str).str.strip()

    # 2. Extract rows specifically for 'Eluasemelaenud'
    # The file has multiple 'EUR' rows, we pull the first instance which is the total portfolio block
    eluase_eur = df[df["Laenuliik_Puhas"] == "EUR"].iloc[0:1].copy()
    eluase_eek = df[df["Laenuliik_Puhas"] == "EEK"].iloc[0:1].copy()

    eluase_eur["Valuuta"] = "EUR"
    eluase_eek["Valuuta"] = "EEK"

    # Combine the filtered blocks into one temporary working table
    koond_df = pd.concat([eluase_eur, eek_row := eluase_eek])

    # 3. Pivot Step (Melt): Convert columns of month names into dynamic rows
    kuud_veerud = [c for c in df.columns if "." in c]

    portfell = pd.melt(
        koond_df,
        id_vars=["Valuuta"],
        value_vars=kuud_veerud,
        var_name="kuu_raw",
        value_name="intress_raw",
    )

    # 4. Data Type Conversion and Cleaning
    # Convert Bank of Estonia format '01.2005' to standard calendar '2005-01-01'
    portfell["kuupaev"] = pd.to_datetime(
        portfell["kuu_raw"], format="%m.%Y"
    ).dt.strftime("%Y-%m-%d")

    # Replace local commas with decimal points and remove % symbol characters (e.g. '3,46%' -> '3.46')
    portfell["intressimaar"] = (
        portfell["intress_raw"]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )

    # Convert everything to floating-point numbers; missing slots coerce safely to NaN
    portfell["intressimaar"] = pd.to_numeric(
        portfell["intressimaar"], errors="coerce"
    )

    # Clean out empty historical rows (like EEK records after the Euro was introduced)
    portfell = portfell.dropna(subset=["intressimaar"])

    # Structure final columns cleanly for relational database use
    lopp_tabel = portfell[["kuupaev", "Valuuta", "intressimaar"]].sort_values(
        by=["kuupaev", "Valuuta"]
    )
    lopp_tabel.columns = ["kuupaev", "valuuta", "intressimaar_laenusummalt"]

    return lopp_tabel.reset_index(drop=True)


# === RUNNING THE SCRIPT ===
df_eluasemelaenud = tootle_eesti_panga_intressid("Intress_EP_raw.csv")

# Save as a clean relational CSV file
df_eluasemelaenud.to_csv("puhastatud_eluasemelaenud_py_final.csv", index=False)

# Preview output matrix
print(df_eluasemelaenud.head(24))

       kuupaev valuuta  intressimaar_laenusummalt
0   2005-01-01     EEK                       3.67
1   2005-01-01     EUR                       3.46
2   2005-02-01     EEK                       3.78
3   2005-02-01     EUR                       3.49
4   2005-03-01     EEK                       3.76
5   2005-03-01     EUR                       3.31
6   2005-04-01     EEK                       3.79
7   2005-04-01     EUR                       3.02
8   2005-05-01     EEK                       3.91
9   2005-05-01     EUR                       3.05
10  2005-06-01     EEK                       4.13
11  2005-06-01     EUR                       3.08
12  2005-07-01     EEK                       4.21
13  2005-07-01     EUR                       3.18
14  2005-08-01     EEK                       4.19
15  2005-08-01     EUR                       3.29
16  2005-09-01     EEK                       4.05
17  2005-09-01     EUR                       3.23
18  2005-10-01     EEK                       4.17
